**Kafka + Unity Catalog variables**

In [0]:
# Kafka connection details

KAFKA_BOOTSTRAP = "pkc-619z3.us-east1.gcp.confluent.cloud:9092"
KAFKA_API_KEY = "2G2K2XOFIL4U3RKG"
KAFKA_API_SECRET = "cfltyyL5v8ozQCaBjGi+Di0ySGi1JXpQNAIsxd9UlYVh1Cv1DdPbhipQbO43krYw"
TOPIC = "atliq.orders.events"

CATALOG = "atliq"
SCHEMA = "streaming"

CKPT = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

**Create catalog/schema/volume**

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}"
)

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.checkpoints"
)

DataFrame[]

**Import functions**

In [0]:
from pyspark.sql import functions as F

**Create Bronze streaming DataFrame**

In [0]:
bronze_df = (
    spark.readStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        KAFKA_BOOTSTRAP
    )
    .option(
        "subscribe",
        TOPIC
    )
    .option(
        "startingOffsets",
        "earliest"
    )
    .option(
        "kafka.security.protocol",
        "SASL_SSL"
    )
    .option(
        "kafka.sasl.mechanism",
        "PLAIN"
    )
    .option(
        "kafka.sasl.jaas.config",
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{KAFKA_API_KEY}" '
        f'password="{KAFKA_API_SECRET}";'
    )
    .load()
)

**Select Bronze columns**

In [0]:
bronze_df = bronze_df.select(
    F.col("key").cast("string").alias("key"),
    F.col("value").cast("string").alias("value"),
    F.col("topic"),
    F.col("partition"),
    F.col("offset"),
    F.col("timestamp")
)

**Start Bronze and WAIT**

In [0]:
bronze_query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        f"{CKPT}/bronze"
    )
    .toTable(
        f"{CATALOG}.{SCHEMA}.bronze_order_events"
    )
)

bronze_query.awaitTermination()

**Verify Bronze**

In [0]:
spark.sql("""
SELECT COUNT(*) AS bronze_events
FROM atliq.streaming.bronze_order_events
""").show()

+-------------+
|bronze_events|
+-------------+
|          718|
+-------------+



**Define Silver schema**

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)

event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_ts", StringType(), True),
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("order_amount", DoubleType(), True),
    StructField("payment_method", StringType(), True)
])

**Read Bronze as streaming source**

In [0]:
bronze_stream = (
    spark.readStream
    .table(
        f"{CATALOG}.{SCHEMA}.bronze_order_events"
    )
)

**Silver transformations**

In [0]:
silver_df = (
    bronze_stream

    .withColumn(
        "event_json",
        F.from_json(
            F.col("value"),
            event_schema
        )
    )

    .select(
        "event_json.*"
    )

    .withColumn(
        "event_ts",
        F.to_timestamp("event_ts")
    )

    .withWatermark(
        "event_ts",
        "10 minutes"
    )

    .dropDuplicates(
        ["event_id"]
    )
)

**Start Silver and WAIT**

In [0]:
silver_query = (
    silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        f"{CKPT}/silver"
    )
    .toTable(
        f"{CATALOG}.{SCHEMA}.silver_order_events"
    )
)

silver_query.awaitTermination()

**Verify Silver**

In [0]:
spark.sql("""
SELECT
    event_type,
    COUNT(*) AS events
FROM atliq.streaming.silver_order_events
GROUP BY event_type
ORDER BY event_type
""").show()

+----------------+------+
|      event_type|events|
+----------------+------+
| order_cancelled|    55|
|    order_placed|   276|
|   order_shipped|   149|
|payment_received|   238|
+----------------+------+



**Read Silver for Gold**

In [0]:
silver_stream = (
    spark.readStream
    .table(
        f"{CATALOG}.{SCHEMA}.silver_order_events"
    )
)

**Gold transformation**

In [0]:
gold_df = (
    silver_stream

    .filter(
        F.col("event_type") == "payment_received"
    )

    # Gold needs its own watermark
    .withWatermark(
        "event_ts",
        "10 minutes"
    )

    .groupBy(
        F.window(
            F.col("event_ts"),
            "5 minutes"
        )
    )

    .agg(
        F.count("*").alias("orders_paid"),
        F.sum("order_amount").alias("revenue")
    )

    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("orders_paid"),
        F.col("revenue")
    )
)

**Start Gold and WAIT**

In [0]:
gold_query = (
    gold_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        f"{CKPT}/gold"
    )
    .toTable(
        f"{CATALOG}.{SCHEMA}.gold_revenue_5min"
    )
)

gold_query.awaitTermination()

**Verify Gold**

In [0]:
spark.sql("""
SELECT
    window_start,
    window_end,
    orders_paid,
    revenue
FROM atliq.streaming.gold_revenue_5min
ORDER BY window_start DESC
LIMIT 12
""").show()

+-------------------+-------------------+-----------+--------+
|       window_start|         window_end|orders_paid| revenue|
+-------------------+-------------------+-----------+--------+
|2026-09-08 10:45:00|2026-09-08 10:50:00|         61|143793.0|
+-------------------+-------------------+-----------+--------+



In [0]:
spark.sql("""
SELECT
    MIN(event_ts) AS first_event,
    MAX(event_ts) AS latest_event,
    COUNT(*) AS total_events
FROM atliq.streaming.silver_order_events
""").show()

+--------------------+--------------------+------------+
|         first_event|        latest_event|total_events|
+--------------------+--------------------+------------+
|2026-09-08 10:47:...|2026-09-14 09:21:...|         718|
+--------------------+--------------------+------------+

